# Detect Columns by using ML

---

In [ ]:
import numpy as np
import pandas as pd
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functions import *
import scipy
import signal
import io
import os
import glob
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from scipy.ndimage import uniform_filter1d
import xgboost as xgb
from tqdm import tqdm

**Machine learning (ML)** is a branch of artificial intelligence. It allows a computer to learn from data and to improve decision making with experience.

---

## Using Random Forest

**L'Arbre de Décision (Decision Tree) :**
Imagine un jeu de "Qui est-ce ?". L'algorithme pose une série de questions par oui/non sur les caractéristiques (features) de tes données pour arriver à une conclusion. Par exemple : "La variance de cette colonne est-elle supérieure à 450 ?" -> Si oui, on va à droite ; si non, on va à gauche.

Le Random Forest repose sur l'apprentissage d'ensemble (Ensemble Learning), et plus précisément sur une technique appelée Bagging (Bootstrap Aggregating). L'algorithme va créer une "forêt" composée de dizaines, voire de centaines d'arbres de décision.

Pour classer une nouvelle colonne (Normale vs Défectueuse), la forêt fait passer les données de la colonne dans tous ses arbres. Chaque arbre vote. La classe qui obtient la majorité des votes l'emporte.

In [ ]:
def load_img_strict(path):
    """
    Lecture universelle et stricte pour garantir que l'entraînement 
    et l'inférence voient EXACTEMENT la même chose.
    """
    # IMREAD_ANYDEPTH force la lecture en 16-bits
    img = cv2.imread(path, cv2.IMREAD_ANYDEPTH)
    
    # Sécurité au cas où l'image aurait été sauvegardée avec 3 canaux (couleur)
    if img.ndim == 3:
        img = img[:, :, 0]
        
    return img

### Features that we use : 

- Moyenne `mean` : Valeur moyenne des intensités de la colonne. Luminosité globale de la colonne. Si trop ou trop basse peut indiquer anomalie.
- Ecart-type `std` : Dispersion autour de la moyenne. Colonne peut-être *noisy* si l'écart-type est très élevé.
- Min et Max : Si la colonne à des valeurs super hautes ou super basses c'est que l'amplificateur est défectueux (dixit Phlypo)
- Range `range` : Différence entre max-min
- Médian `median` : Valeur centrale. Comparaison avec la moyenne intéressant.
- Quantile `q25` `q75` : Quartile.
- Skewness `skewness` : Asymétrie 
- Kurtosis `kurtosis`: Applatissement 
- Energie `energy` : Energie d'un signal (dixit Signals and Systems)
- Entropie `entropy` : J'ai pas vrmt compris / Mesure du désordre ou de l'incertitude dans la distribution des intensités.
- Nombres de pics `num_peaks` : les pics dans la colonne si gros peut-être *fragmented*
- Moyenne du gradiant `mean_gradient` : Moyenne des différences entre pixels consécutifs

#### A faire : 
- Différence Spatiale (un peu comme mes autres méthodes)
- Différence Temporelle (pour les blinking)

In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter1d

def extract_image_features_vectorized(img_prev, img_curr, img_next):
    # Conversion en float64 pour toute l'image d'un coup
    img_prev = img_prev.astype(np.float64)
    img_curr = img_curr.astype(np.float64)
    img_next = img_next.astype(np.float64)
    
    height, width = img_curr.shape[:2]
    
    # DÉTECTION DE LA RÉSOLUTION (Pour la fenêtre du LT_moy)
    if height < 700: 
        taille_fenetre = 18
    elif height < 1050: 
        taille_fenetre = 31
    else: 
        taille_fenetre = 24

    # --- Statistiques Globales ---
    col_mean = np.mean(img_curr, axis=0) # Vecteur avec la moyenne de chaque colonne
    # col_energy = np.sum(img_curr ** 2, axis=0) / height

    # 1. LT_moy (méthode développé avant)
    tendance_locale = uniform_filter1d(col_mean, size=taille_fenetre, mode='reflect')

    mean_sq = np.mean(img_curr ** 2, axis=0)
    tendance_sq = uniform_filter1d(mean_sq, size=taille_fenetre, mode='reflect')
    std_locale = np.sqrt(np.maximum(tendance_sq - tendance_locale**2, 0))
    
    ecart_lt_moy = np.abs(col_mean - tendance_locale)
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5)
    
    # Tentative Spatial & Temporel

    # On regarde autoour
    left_neighbor = np.roll(col_mean, 1)
    right_neighbor = np.roll(col_mean, -1)
    neighbor_mean = (left_neighbor + right_neighbor) / 2.0
    
    # Pour gérer les bords 
    neighbor_mean[0] = col_mean[1]
    neighbor_mean[-1] = col_mean[-2]
    
    spatial_diff = np.abs(col_mean - neighbor_mean)
    
    # Contexte temporel
    mean_prev = np.mean(img_prev, axis=0)
    mean_next = np.mean(img_next, axis=0)
    diff_temp_absolue = np.abs(col_mean - mean_prev)
    scintillement_temporel = np.abs(col_mean - ((mean_prev + mean_next) / 2.0))

    # Final features
    features_matrix = np.column_stack((
        ecart_lt_moy, ratio_lt_moy,
        spatial_diff, diff_temp_absolue, scintillement_temporel,
        # col_mean, col_energy
    ))
    
    return features_matrix

def process_single_image(img_num, images_list, json_data):
    """Prépare les labels et lance l'extraction de l'image entière"""
    img_curr = images_list[img_num]
    img_prev = images_list[img_num - 1] if img_num > 0 else img_curr
    img_next = images_list[img_num + 1] if img_num < (len(images_list) - 1) else img_curr
    
    width = img_curr.shape[1]
    
    # Extraction vectorisée
    features_matrix = extract_image_features_vectorized(img_prev, img_curr, img_next)
    
    # Création rapide des labels (y)
    defects = set(get_defect_coordinates(json_data, img_num))
    y_img = [1 if x in defects else 0 for x in range(width)]
    
    return features_matrix.tolist(), y_img

In [ ]:
def build_dataset(images, json_data):
    print(f"Lancement de l'extraction sur {len(images)} images en parallèle...")
    
    # n_jobs=-1 (utilisation de tous les coeurs)
    # return_as="generator" permet à tqdm de se mettre à jour en temps réel
    result_generator = Parallel(n_jobs=-1, require="sharedmem", return_as="generator")(
        delayed(process_single_image)(img_num, images, json_data) 
        for img_num in range(len(images))
    )
    
    X = []
    y = []
    
    # Visuel cool
    for X_img, y_img in tqdm(result_generator, total=len(images), desc="Extraction Multicoeur"):
        X.extend(X_img)
        y.extend(y_img)
        
    return np.array(X), np.array(y)

### Lezz gooo

In [ ]:
def load_images(folder='train', type='VGA', sequence='sequence_1', dyn='low dyn with columns 1'):
    """
    Version pour ML
    """
    images = []
    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')
    fichiers_trouves = sorted(glob.glob(chemin_recherche))
    
    for image_path in fichiers_trouves:
        img = cv2.imread(image_path, cv2.IMREAD_ANYDEPTH)
        
        if img is None:
            print(f"Impossible de charger : {image_path}")
            continue
            
        if img.ndim == 3:
            img = img[:, :, 0]
            
        images.append(img)
        
    return images

In [ ]:
# MEGA DATASET 

CAMERA_TYPE = 'SXGA'  # Change ceci en 'HD', 'VGA' ou 'SXGA'

# Création automatique du nouveau dossier demandé
os.makedirs('new_models_2', exist_ok=True)

dyn_mapping = {
    1: 'low dyn with columns 1',
    2: 'low dyn with columns 2',
    3: 'low dyn with columns 3'
}

# On prépare deux listes distinctes pour le Train et le Test
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

noms_colonnes = [
    'ecart_lt_moy', 'ratio_lt_moy', 
    'spatial_diff', 'diff_temp_absolue', 'scintillement_temporel', 
    # 'mean', 'energy'
]

# =========================================================================
# 2. BOUCLE D'EXTRACTION GLOBALE (Séquences 1, 2 et 3 réunies !)
# =========================================================================
X_train_list = []
y_train_list = []

for seq in [1, 2, 3]:  # On prend TOUT pour le modèle final
    for config in [1, 2, 3]:
        print(f"\n Pour le Mordor : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        dossier_json = r"C:\Users\alexc\Documents\Data_challenge\Data_Challenge\results"
        json_path = os.path.join(dossier_json, f"{CAMERA_TYPE}_{sequence_name}_config_{config}.json") 
        
        try:
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            X_batch, y_batch = build_dataset(images, json_data)
            
            # Tout le monde va dans l'entraînement global
            X_train_list.append(X_batch)
            y_train_list.append(y_batch)
            
        except Exception as e:
            print(f"Erreur pour Seq {seq} / Config {config} : {e}")
            continue

X_train_raw = np.vstack(X_train_list)
y_train_raw = np.concatenate(y_train_list)

print(f"\nExtraction terminée / Taille globale du dataset : {X_train_raw.shape}")

# Rééquilibrage car trop de COLONNES SAINES
indices_defauts = np.where(y_train_raw == 1)[0]
indices_sains = np.where(y_train_raw == 0)[0]

nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_train_balanced = X_train_raw[indices_finaux]
y_train_balanced = y_train_raw[indices_finaux]

# Fitting of the model
X_train = pd.DataFrame(X_train_balanced, columns=noms_colonnes)
y_train = y_train_balanced

ratio_poids = float(np.sum(y_train == 0)) / np.sum(y_train == 1) # punition

clf = xgb.XGBClassifier(
    n_estimators=150,
    scale_pos_weight=ratio_poids, 
    random_state=42,
    tree_method="hist",
    device="cpu"
)

clf.fit(X_train, y_train)

# Sauvegarde dans 'new_models'
model_path = f'new_models_2/xgboost_{CAMERA_TYPE.lower()}_baseline.json'
clf.save_model(model_path)
print(f"MODÈLE ULTIME SAUVEGARDÉ : {model_path} !")

In [ ]:
y_pred = clf.predict(X_train)

print(classification_report(y_train, y_pred))